# Phase II, Pair 1: question D — front days inside the model

*What differs inside the model between a normal day and a front day, and can the state predict that the forecast will fail?*

| | |
|---|---|
| Authors | |
| Status | in progress |
| Phase I result this builds on | Sandro's front runs (30 front starts against 306 normal starts: about 5.3 °C too warm 3 h after the front, band holds 32 % against 81 % in normal weather) and Exp 11 (band honest on normal days, half the hours below the band with a front coming) |

## 1 · Question

What differs inside the model between a normal day and a front day, and can the state predict that the forecast will fail?

Two words need a fixed meaning for the whole notebook:

- **Front day** (the "special" case): a forecast start 12 h or 6 h before the peak of a cold front in `fronts.csv`
  (`cold_front == True`), i.e. the `front-12` and `front-6` starts of Phase I. Normal starts whose forecast window or last
  day of context contains a `cold_front` passage at that point in `events.parquet` are neither normal nor front: they are
  kept out of the normal reference.
- **Fails**: the mean absolute error of the median over the 24 forecast hours is above **2 K**. (Phase I counted hours with
  an error above 2 K; the band-miss definition is not used for "fails", band coverage is reported on its own.)

## 2 · Why this question

Phase I showed *that* the forecast breaks at fronts: the error rises to 5–7 °C and the 10–90 % band holds only 32 % of
the hours, against 81 % in normal weather. The band does not widen before a front, so the output does not show that the
model is unsure. We do not know whether anything *inside* the model is different on those days. If the state before a
front already sits apart from the states of normal days, the model "has" the information but does not turn it into a
wider band. If it does not, the failure is invisible to the model.

## 3 · Hypothesis and what would count as an answer

**Hypothesis.** Front states sit apart from normal states, and the distance from the normal states goes with failure.

- **If the hypothesis is right, we would see:** front states farther from their nearest normal states than normal states
  are from each other (also against temperature-matched normal cases), a positive rank correlation between that distance
  and the error, a higher fail rate and lower band coverage in the far distance bins, and a distance that separates
  failures from successes (AUC clearly above 0.5) when splitting by point and by month.
- **If it is wrong, we would see:** front and normal states mixed (same distance distribution), no relation between
  distance and error, and coverage that does not depend on the distance: the failures are invisible in the state.

## 4 · Setup

*Imports, paths, model loading.*

### The "(check)" points (source: `tirex-2==0.3.1` as pinned, and `src/tirex2/model/tirex2.py` on `main`)

- **`input_patch_size`**: set by the checkpoint's config, printed below. Phase I Exp 10 found the important hours repeating
  every 32 h, which points to patches of 32 h.
- **Number of blocks**: `len(core.stack)`, printed below; one `MultivariateBlock` (time mixer, then variate attention)
  per entry of the config's `stack_config.recipe`.
- **`_forward_layerwise`**: `_forward_layerwise(self, x, group_vector, target_mask, known_covariate_mask)` in 0.3.1;
  `main` adds `indices=None`. It is a generator that yields the tokens after each block, `[rows, tokens, dim]`. We do not
  call it ourselves: forward hooks on `core.stack[b]` see the same tensors.
- **Grad enabled**: the forward pass itself has no `no_grad`, but `TiRex2.predict` (and so `model.forecast`) is decorated
  with `@torch.no_grad`. `_predict(..., preserve_grad=True)` keeps the graph. D and E do not need gradients.

### The state

**State = output of the last block at the last context patch** (the token holding the newest observed hours), for the row
of the point. The input is NaN-padded on the right by `future_len`, so the last tokens are the forecast window and the
last *context* token is `L - future_len // input_patch_size - 1`. With sign-flip TTA, `predict` runs twice; the state is
taken from the first (unflipped) pass.

The function lives in `model_state.py` next to the notebooks. **E imports the same function**, so both questions use
exactly the same state.

In [ ]:
import os
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from model_state import context_token, hidden_states, state_and_forecast
from tirex2 import load_model

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DATA = Path(os.environ.get("AIVIS_DATA", ROOT / "data" / "austria_15km"))
CKPT = os.environ.get("TIREX_CKPT", "NX-AI/TiRex-2")
DEVICE = os.environ.get("TIREX_DEVICE", "cpu")
RESULTS = ROOT / "data" / "notebooks" / "results" / "04_question_d"
RESULTS.mkdir(parents=True, exist_ok=True)

CONTEXT, HORIZON, SEED = 672, 24, 0

model = load_model(CKPT, device=DEVICE)  # as in phase1-model-behavior/pair-1/run_forecasts.py
core = model.model
h = hidden_states(model, np.zeros((1, CONTEXT), np.float32))
print(f"input_patch_size {core.input_patch_size} | blocks {len(core.stack)} | embedding_dim {core.embedding_dim}")
print(
    f"tokens per input {h.shape[2]} | state token {context_token(core, h.shape[2])} | context_len {core.context_len}"
    f" | future_len {core.future_len} | tta_sign_flip {core.tta_sign_flip}"
)

## 5 · Data

Phase I design: 672 h context, 24 h horizon, start times and points of `tirex2_forecasts.npz` (336 starts × 378 points:
306 normal grid starts and 30 front starts). The context is filled exactly as in `run_forecasts.py`.

`FULL = True` uses all 336 × 378 cases (about 75 min on CPU, as in Phase I). The default is a sample:
**all front starts**, plus for each front start `N_MATCH` normal starts at the nearest hour of day and the nearest date
(at least 3 days away from any cold-front peak), at `N_POINTS` points spread over Austria (farthest-point sampling).
Matching by season and hour of day keeps the daily cycle and the season from separating the groups on their own.

The states (and the forecasts computed with them) are saved to `data/notebooks/results/04_question_d/`, not to git.

In [ ]:
FULL, N_MATCH, N_POINTS = False, 2, 120
FAIL_K = 2.0  # "fails": MAE of the median over 24 h above this (K)

pts = pd.read_csv(DATA / "points_with_labels.csv").set_index("point_id")
grid = pd.read_parquet(DATA / "grid_long.parquet", columns=["point_id", "time", "T2M"])
OBS = grid.pivot(index="time", columns="point_id", values="T2M")
T = OBS.interpolate(limit_area="inside").ffill().bfill()  # as run_forecasts.py
times, pids = T.index, T.columns.to_numpy()
SER = T.to_numpy(np.float32)
OBSV = OBS.to_numpy(np.float32)
fronts = pd.read_csv(DATA / "fronts.csv", parse_dates=["peak_time"])
front_peaks = times.get_indexer(pd.to_datetime(fronts[fronts.cold_front].peak_time, utc=True))

P1 = np.load(DATA / "tirex2_forecasts.npz", allow_pickle=True)
tags = P1["tags"].astype(str)
origin = P1["origin_idx"].astype(int)
p1_col = {p: k for k, p in enumerate(P1["pids"])}
assert (P1["times"] == times.astype("int64").to_numpy()).all(), "time axis differs from the Phase I file"

front_k = np.where(np.char.startswith(tags, "front"))[0]
grid_k = np.where(tags == "grid")[0]
if FULL:
    start_k = np.r_[grid_k, front_k]
else:
    far = grid_k[[np.abs(front_peaks - origin[k]).min() >= 72 for k in grid_k]]
    hour = lambda k: times[origin[k]].hour
    match = set()
    for k in front_k:
        cand = sorted(
            far, key=lambda g: (min(abs(hour(g) - hour(k)), 24 - abs(hour(g) - hour(k))), abs(origin[g] - origin[k]))
        )
        match.update(int(g) for g in cand[:N_MATCH])
    start_k = np.r_[np.sort(list(match)).astype(int), front_k]

XY = pts.loc[pids, ["x", "y"]].to_numpy(float) / 1000
if FULL or N_POINTS >= len(pids):
    chosen = np.arange(len(pids))
else:
    rng = np.random.default_rng(SEED)
    chosen = [int(rng.integers(len(pids)))]
    while len(chosen) < N_POINTS:
        d = np.linalg.norm(XY[:, None] - XY[chosen][None], axis=-1).min(axis=1)
        chosen.append(int(np.argmax(d)))
    chosen = np.array(chosen)

cases = pd.DataFrame(
    [(int(k), origin[k], tags[k], int(j), pids[j]) for k in start_k for j in chosen],
    columns=["p1_k", "i", "tag", "j", "pid"],
)
print(f"{len(start_k)} starts ({np.isin(start_k, front_k).sum()} front) x {len(chosen)} points = {len(cases)} cases")

In [ ]:
path = RESULTS / "states.npz"
key = cases[["i", "pid"]].to_numpy()
if path.exists() and np.array_equal(np.load(path)["key"], key):
    z = np.load(path)
    STATES, FRESH = z["states"], z["preds"]
else:
    t0 = time.time()
    ctx = np.stack([SER[i - CONTEXT : i, j] for i, j in zip(cases.i, cases.j)])
    STATES, FRESH = state_and_forecast(model, ctx, HORIZON, batch_size=128)
    np.savez_compressed(path, key=key, states=STATES, preds=FRESH)
    print(f"states for {len(cases)} cases in {time.time() - t0:.0f}s")
print("states", STATES.shape)

## 6 · Technique 1

*What it shows, and what it cannot show.*

## 7 · Technique 2

*What it shows, and what it cannot show.*

## 8 · Comparing the techniques

*Do they agree? Where do they differ?*

## 9 · Checks

*Does the result hold on other dates, other points, other groups? What did we try to break it with?*

## 10 · Figure decisions

| Figure | Keep / change / drop | Why |
|---|---|---|

## 11 · What it means

*Answer to the question, in plain words. What is still open?*

## 12 · What did not work

*Dead ends and why. They are part of the record.*